# Обучение модели ResNet18 для классификации AI vs Human Generated Images

Этот notebook содержит код для обучения модели ResNet18 на датасете Train_1 и оценки качества на Test_1.

## 1. Импорт библиотек

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Рабочая директория

Технические файлы проекта сложены в `mlops_pipeline`, поэтому перед запуском следующих ячеек переходим в эту папку.


In [1]:
%cd mlops_pipeline


/Users/anastasiasergeeva/Documents/New project/hw4_big_data_mlops/mlops_pipeline


## 2. Подготовка данных

In [2]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def _resolve_path(self, file_name):
        raw_path = Path(str(file_name))
        candidates = [
            self.root_dir / raw_path,
            self.root_dir / raw_path.name,
            self.root_dir / "train_data" / raw_path.name,
            self.root_dir / "test_data" / raw_path.name,
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        raise FileNotFoundError(f"Cannot find {file_name}; tried: {candidates}")
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = self._resolve_path(row['file_name'])
        image = Image.open(img_path).convert('RGB')
        label = int(row['label'])
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Train_1/train.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Test_1/test.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Test_1',
    transform=test_transform
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

### Проверка размеров split-ов

Эта ячейка добавлена к блоку подготовки данных, чтобы явно показать, какие данные использовались для обучения и дообучения.

In [3]:
for name, csv_path in {
    "Train_1": "ai-vs-human-generated-dataset-hw/Train_1/train.csv",
    "Test_1": "ai-vs-human-generated-dataset-hw/Test_1/test.csv",
    "Train_2": "ai-vs-human-generated-dataset-hw/Train_2/train.csv",
    "Test_2": "ai-vs-human-generated-dataset-hw/Test_2/test.csv",
}.items():
    df = pd.read_csv(csv_path)
    print(name, "rows=", len(df), "labels=", df["label"].value_counts().sort_index().to_dict())

Train_1 rows= 9993 labels= {0: 4994, 1: 4999}
Test_1 rows= 3997 labels= {0: 1996, 1: 2001}
Train_2 rows= 3997 labels= {0: 1960, 1: 2037}
Test_2 rows= 2000 labels= {0: 1005, 1: 995}


## 3. Создание модели ResNet18

In [ ]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print(f'Model architecture:')
print(model)

## 4. Определение функции потерь и оптимизатора

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 5. Функция обучения

In [4]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    total = 0
    
    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    return epoch_loss, epoch_acc, epoch_f1

## 6. Функция валидации

In [5]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total += batch_size
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)
    epoch_precision = precision_score(all_labels, all_preds, zero_division=0)
    epoch_recall = recall_score(all_labels, all_preds, zero_division=0)
    
    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 7. Обучение модели

In [6]:
import json
base_metrics = json.loads(Path("artifacts/base/metrics.json").read_text())
print("S3 artifact:", base_metrics["s3_uri"])
print("Last train metrics:", base_metrics["train_history"][-1])
print("Test metrics:", base_metrics["test_metrics"])

S3 artifact: s3://hw4-models/models/resnet18_base_train1.pt
Last train metrics: {'epoch': 5, 'accuracy': 0.9485639947963574, 'f1': 0.9486410871302957, 'precision': 0.9476941505290477, 'recall': 0.9495899179835967, 'loss': 0.13411892452301008, 'lr': 0.0001}
Test metrics: {'accuracy': 0.9527145359019265, 'f1': 0.9533908754623921, 'precision': 0.9410905550146057, 'recall': 0.9660169915042479, 'loss': 0.1265742069099556}


In [7]:
print("TensorBoard event files:")
for path in sorted(Path("tb_logs").glob("*/*")):
    print(path)

TensorBoard event files:
tb_logs/base_train1/events.out.tfevents.1778764843.produniversitygpu103939z504.2133658.0
tb_logs/base_train1/events.out.tfevents.1778765216.produniversitygpu103939z504.2134645.0
tb_logs/finetuned_train2/events.out.tfevents.1778765372.produniversitygpu103939z504.2135422.0


## 8. Оценка модели на тестовом датасете

In [ ]:
# Добавьте логику оценки модели на тестовом датасете, как метрику в tensorboard
# pip install tensorboard
# команда для поднятия
# tensorboard --logdir=my_logs

### Итоговая оценка базовой модели на Test_1

Метрики базовой модели сохранены в `artifacts/base/metrics.json` и также залогированы в TensorBoard

In [8]:
pd.DataFrame([base_metrics["test_metrics"]])

   accuracy        f1  precision    recall      loss
0  0.952715  0.953391   0.941091  0.966017  0.126574


## 9. Дообучите модель на втором датасете и постройте DVC пайплайн

In [ ]:
# Добавьте логику дообучения
# Добавьте PVC пайплайн

DVC pipeline описан в `dvc.yaml`. Он воспроизводит полный MLOps-сценарий: обучение базовой модели, скачивание базовой модели из S3, дообучение на `Train_2`, оценку на `Test_2`, загрузку новой версии модели в S3 и сравнение результатов

In [9]:
# Основные команды запуска:
# dvc init --no-scm
# dvc repro
# dvc dag > reports/dvc_dag.txt

print(Path("reports/dvc_dag.txt").read_text())

                      +-----------------------+          
                      | train_base_and_upload |          
                      +-----------------------+          
                      ***                   ****         
                  ****                          ***      
                **                                 ****  
+-----------------------+                              **
| download_base_from_s3 |                               *
+-----------------------+                               *
            *                                           *
            *                                           *
            *                                           *
 +---------------------+                               **
 | finetune_and_upload |                           ****  
 +---------------------+                        ***      
                      ***                   ****         
                         ****           ****             
              

In [10]:
finetuned_metrics = json.loads(Path("artifacts/finetuned/metrics.json").read_text())
print("S3 artifact:", finetuned_metrics["s3_uri"])
print("Last train metrics:", finetuned_metrics["train_history"][-1])
print("Test metrics:", finetuned_metrics["test_metrics"])

S3 artifact: s3://hw4-models/models/resnet18_finetuned_train2.pt
Last train metrics: {'epoch': 3, 'accuracy': 0.9577182887165374, 'f1': 0.9587099926704129, 'precision': 0.9542801556420234, 'recall': 0.9631811487481591, 'loss': 0.12030760262768239, 'lr': 1e-05}
Test metrics: {'accuracy': 0.9595, 'f1': 0.9591939546599496, 'precision': 0.9616161616161616, 'recall': 0.9567839195979899, 'loss': 0.11851048466563224}


In [11]:
print(Path("reports/comparison.md").read_text())

# Model comparison

| version          |   train_loss |   train_accuracy |   train_f1 |   test_loss |   test_accuracy |   test_f1 |   test_precision |   test_recall |
|:-----------------|-------------:|-----------------:|-----------:|------------:|----------------:|----------:|-----------------:|--------------:|
| base_train1      |     0.134119 |         0.948564 |   0.948641 |    0.126574 |        0.952715 |  0.953391 |         0.941091 |      0.966017 |
| finetuned_train2 |     0.120308 |         0.957718 |   0.95871  |    0.11851  |        0.9595   |  0.959194 |         0.961616 |      0.956784 |

## Delta finetuned - base

|                |       delta |
|:---------------|------------:|
| train_loss     | -0.0138113  |
| train_accuracy |  0.00915429 |
| train_f1       |  0.0100689  |
| test_loss      | -0.00806372 |
| test_accuracy  |  0.00678546 |
| test_f1        |  0.00580308 |
| test_precision |  0.0205256  |
| test_recall    | -0.00923307 |


## 10. Напишите вывод о полученных результатах

### Вывод

Базовая модель `base_train1` была обучена на `Train_1` и оценена на `Test_1`. После этого модель была скачана из S3 и дообучена на `Train_2`; новая версия `finetuned_train2` оценена на `Test_2` и загружена в S3.

По сохраненным метрикам дообученная версия показывает лучшее качество на своем тестовом split-е: `test_accuracy` вырос с `0.9527` до `0.9595`, `test_f1` вырос с `0.9534` до `0.9592`, `test_loss` снизился с `0.1266` до `0.1185`. Precision вырос заметно (`0.9411 -> 0.9616`), recall немного снизился (`0.9660 -> 0.9568`), то есть после fine-tuning модель стала осторожнее относить изображения к positive-классу.